In [ ]:
import cartopy.crs as ccrs
import numpy as np
import xarray as xr
import cmaps
import glob
from REMO2020_mod import modelclass, open_mfdataset, eobs_preci_open, calc_remo_precipitation, domainclass, plotclass, remo_plotter

In [ ]:
from dask.distributed import Client, progress
import dask
dask.config.set({'temporary_directory': '/scratch/g/g300096/dask/tmp'})
client = Client()
client

In [ ]:
# define the runs to be plotted
modelruns =[] # init

modelruns.append(modelclass("036041","REMO2020",True,varname='Dtemp2'))
modelruns.append(modelclass("036043","REMO2020 TEB",True,varname='Dtemp2'))
modelruns.append(modelclass("036042","REMO2020 iMOVE",True,varname='Dtemp2'))
modelruns.append(modelclass("036044","REMO2020 TEB iMOVE",True,varname='Dtemp2'))

In [ ]:
# give path, prefix and suffix (the main name for the plots)
figpath = './figures/'
prefix = 'eobs_preci_abs_seas_'
suffix = 'REMO2020_main'
# or give absolute figure name (leave otherwise empty (''))
afigname = 'fig4'

In [ ]:
# REMO and EOBS datapath
rdata = './data_in/model_data/'
edata = './data_in/measurement_data/E-OBS/'

In [ ]:
# define star year and end year
ystart=2011
yend=2020

In [ ]:
# Define the order of seasons to be used (also when plotting)
seasons = ["DJF","MAM","JJA","SON"]

In [ ]:
# Load the lake mask
lake_mask = xr.open_dataset(rdata+"FLake_input/flake_v3_glcc_defD10.0m_frac_EUR-11.nc")["FLFRA"]

In [ ]:
# E-OBS version
eobv='v33.0e'

In [ ]:
# Eobs file name
eobsfile = edata+eobv+"/day/var/rr/rr_ens_mean_0.1deg_reg_"+eobv+".nc"

In [ ]:
# set the minimum number days with data per month to be included
dlim=21

In [ ]:
# lake fraction cut-off limit
clake = 0.5

In [ ]:
# Target grid
trggrid = "EUR-11"

In [ ]:
# calculate eobs variables
eobs_mdata, final_mask, ndays_ps = eobs_preci_open(eobsfile,ystart,yend,trggrid,dlim,lake_mask,clake)

In [ ]:
yearlist = np.arange(ystart,yend+1)
for obj in modelruns:
    obj.filenames = []
    for year in yearlist:
        pattern = rdata+obj.userexp+"/"+str(year)+"/e"+obj.userexp+"m*.nc"
        obj.filenames += glob.glob(pattern)
    obj.filenames.sort()

In [ ]:
# Set halo zone length (will be cutted out)
halo = 8

In [ ]:
# calculate the precipitation difference to EOBS
ii = 0
for obj in modelruns:   
    ds = open_mfdataset(obj.filenames, parallel=False, chunks='auto')
    precip, dpsdummy = calc_remo_precipitation(ds)
    # Merge data (get identical coordinates)
    precip_all = xr.merge([precip.rename("totpre"), eobs_mdata.rename("eobs_m_data"),final_mask.rename("mask")], compat="override", join="override")
    # apply masking (eobs+lake) and subtract the values, finally making seasonal means
    pre_mod_tmp = precip_all.totpre.where(precip_all.mask,drop=False).groupby("time.season").sum(skipna=True)
    if(ii == 0):
        final_mask_seas = precip_all.mask.groupby("time.season").sum(skipna=True) #.sum(dim="time").compute()
        pre_eobs_abs = precip_all.eobs_m_data.where(precip_all.mask,drop=False).groupby("time.season").sum(skipna=True).squeeze()
        # small adjustment to zero values (occur near borders)
        final_mask_seas = final_mask_seas.where(pre_eobs_abs > 0, drop=False)
        pre_eobs_abs = pre_eobs_abs.where(final_mask_seas > 0, drop=False)
        # calculate dps
        tmp_mask = xr.merge([pre_eobs_abs, ndays_ps.rename("mask")], compat="override", join="override")
        dps = tmp_mask.mask.where(final_mask_seas > 0, drop=False)
        # change pre_eobs_abs units from mm/season -> mm/day
        pre_eobs_abs = (pre_eobs_abs / dps).compute()
    # get identical coordinates
    final_tmp = xr.merge([pre_mod_tmp.rename("totpre"),dps.rename("dps"),pre_eobs_abs.rename("pre_eobs_abs"),final_mask_seas.rename("final_mask_seas")], compat="override", join="override")
    obj.vardiff = (final_tmp.totpre/final_tmp.dps - final_tmp.pre_eobs_abs).where(final_tmp.final_mask_seas, drop=False).squeeze()
    obj.vardiff = obj.vardiff.isel(rlat=slice(halo,obj.vardiff.sizes["rlat"]-halo), rlon=slice(halo,obj.vardiff.sizes["rlon"]-halo)).compute()
    # not used, set to zero
    obj.mask = np.zeros(len(seasons))
    # increase counter
    ii = ii + 1
    #t_stat
    if(obj.r2020file):
        pole = (ds.rotated_latitude_longitude.grid_north_pole_longitude,ds.rotated_latitude_longitude.grid_north_pole_latitude)
        rotated_pole = ccrs.RotatedPole(pole[0], pole[1])
        extent = ([ds.rlon.min(), ds.rlon.max(), ds.rlat.min(), ds.rlat.max()], rotated_pole)
        
    # clean memory
    del ds, precip, dpsdummy, precip_all, pre_mod_tmp, final_tmp
# Clean memory    
del tmp_mask
# cut also halo so domain mean will be comparable
pre_eobs_abs = pre_eobs_abs.isel(rlat=slice(halo,pre_eobs_abs.sizes["rlat"]-halo), rlon=slice(halo,pre_eobs_abs.sizes["rlon"]-halo)).compute()

In [ ]:
# define the domains to be plotted
domaininfo =[] # init
domaininfo.append(domainclass("EU",[-22.5,47.5,30,72.5], hspace=0.35))

In [ ]:
# define plotclasses for absolute and difference (model) plots
absplotc = plotclass(label='Precipitation [mm/day]', lspace=np.linspace(0,1,10), vmin=0, vmax=5, linvert=False, cmapp=cmaps.GMT_drywet)
modplotc = plotclass(label='$\Delta$Precipitation [mm/day]', lspace=np.linspace(0,1,30), vmin=-3, vmax=3, linvert=True, cmapp=cmaps.BlueWhiteOrangeRed)

In [ ]:
# Loop over domains
for dom in range(len(domaininfo)):
    # Create figure name
    figname=prefix+domaininfo[dom].name+'_'+suffix
    # check if we want to use absolute figure name
    if(afigname):
        figname = afigname
    # Call plotter
    remo_plotter(absplotc,modplotc,domaininfo[dom],modelruns,"E-OBS",pre_eobs_abs,seasons,pole,False,figpath,figname)